### Step 0: Import libraries, read data and display head

In [67]:
import numpy as np
import pandas as pd

df = pd.read_csv("./DirtyData.csv")
df.head(20)

,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.34,NaN
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,NaN,24115.00
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.77,9378.72
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.77,7200.12
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.49,13295.77
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,NaN,14493.00
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.02,15756.15
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Woman,236589.68,35488.45
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Woman,194301.63,29145.24
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.05,NaN


### Step 1: Inspect before touching anything

In [68]:
print("Shape:", df.shape)
print("\nMissing values per column:", df.isna().sum())
print(f"\nNegative incomes: {(df['income'] < 0).sum()}")
print(f"\nNegative taxes: {(df['tax_15'] < 0).sum()}")
print(f"Gender categories and their columns: {df['gender'].value_counts()}")

Shape: (1000, 6)

Missing values per column: first_name      0
last_name       0
email           0
gender          0
income        115
tax_15         28
dtype: int64

Negative incomes: 86

Negative taxes: 0
Gender categories and their columns: gender
Female         469
Male           409
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Man             10
Woman            3
Men              3
Women            2
Name: count, dtype: int64


### Step 2: Handle missing value by Reconstruction (not guessing)
Mean imputation would fill a missing income with the average income. But we can do better because we know tax_15 is always exactly 15% of the income. 

In [69]:
df["income"] = np.where(df["income"].isna() & df["tax_15"].notna(), df["tax_15"] * (100/15), df["income"])
df["tax_15"] = np.where(df["tax_15"].isna() & df["income"].notna(), df["income"] * 0.15, df["tax_15"])

print(f"Missing value after reconstruction: {df.isna().sum()}")

Missing value after reconstruction: first_name    0
last_name     0
email         0
gender        0
income        0
tax_15        0
dtype: int64


### Step 3: fix negative incomes

In [70]:
df["income"] = df["income"].abs()
df["tax_15"] = df["income"] * 0.15

print(f"Negative incomes remaining: {(df["income"] < 0).sum()}")

Negative incomes remaining: 0


### Step 4: Standardize `gender` with Judgement

In [71]:
df["gender"].value_counts()

gender
Female         469
Male           409
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Man             10
Woman            3
Men              3
Women            2
Name: count, dtype: int64

In [72]:
gender = {
  "Man": "Male", "Men": "Male",
  "Woman": "Female", "Women": "Female"
}

df["gender"] = df["gender"].replace(gender)
print(f"Unique gender value counts: {df["gender"].value_counts()}")

Unique gender value counts: gender
Female         474
Male           422
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Name: count, dtype: int64


### Step 5: Normalize using min-max

In [73]:
# rescaling columns from 0 to 1 [0, 1]
min = df["income"].min()
max = df["income"].max()

income_range = max - min

df["income_norm"] = (df["income"] - min) / income_range
print("Income range:", round(min, 2), "to", round(max, 2))

print("Normalized range:", round(df["income_norm"].min(), 3), "to", round(df["income_norm"].max(), 3))

df[["income", "income_norm"]].head(20)

Income range: 15245.38 to 326986.67
Normalized range: 0.0 to 1.0


,income,income_norm
0,123072.340000,0.345886
1,160766.666667,0.466801
2,62524.770000,0.151662
3,48000.770000,0.105072
4,88638.490000,0.235430
5,96620.000000,0.261033
6,105041.020000,0.288045
7,236589.680000,0.710026
8,194301.630000,0.574375
9,239219.050000,0.718460


### Step 6: Final check and save

In [74]:
print("Missing values remaining:", df.isna().sum())
print("Negative income values remaining:", (df["income"] < 0).sum())

# save it do a new csv
df.to_csv("./CleanedIncomeData.csv", index=False)
print("\nsaved cleaned data to new csv")

Missing values remaining: first_name     0
last_name      0
email          0
gender         0
income         0
tax_15         0
income_norm    0
dtype: int64
Negative income values remaining: 0

saved cleaned data to new csv
